# Code for doing the different clustering methods, as well as data preprocessing

In this file we run the different methods

### DMACN - Deep Multi-kernel Auto-encoder Clustering Network
First load the data

In [3]:
from scipy.io import loadmat
import torch

# Load the data
PTSD = loadmat("C:\\Users\\oddar\\Downloads\\PTSD_connectivity.mat")
# PTSD is a dataset containing 87 samples (subjects) with 340 features (as vectorized functional connectivity matrices)
# The expected number of clusters are 3

# Define the functional connectivity matrix (example)
Functional_connectivity_matrix = PTSD["connectivities"]  # Example matrix

PTSD_tensor = torch.from_numpy(Functional_connectivity_matrix).float()

# Dummy data
X = PTSD_tensor  # [N,d] float tensor

Define the autoencoder specs

In [4]:
from DMACN import DMACN, DMACNConfig

kernel_specs = [
    {"kind": "rbf", "t": 0.01},
    {"kind": "rbf", "t": 0.05},
    {"kind": "rbf", "t": 0.1},
    {"kind": "rbf", "t": 1},
    {"kind": "rbf", "t": 10},
    {"kind": "rbf", "t": 50},
    {"kind": "rbf", "t": 100},
    {"kind": "poly", "a": 0, "b": 2},
    {"kind": "poly", "a": 0, "b": 4},
    {"kind": "poly", "a": 1, "b": 2},
    {"kind": "poly", "a": 1, "b": 4}
]  # h = 3

cfg = DMACNConfig(
    C=3,  # number of clusters
    dims_enc=[340, 285, 240, 202, 170],   # mid = 2 encoder Linear layers = L/2
    dims_dec=[170, 202, 240, 285, 340],
    kernel_specs=kernel_specs,
    m_fuzz=1.08,
    lam1=0.5,
    lam2=0.5,
    lr=1e-3,
    epochs=500,
    mk_max_iters=20,
    mk_eps_stop=1e-5,
    renormalize_omega_sum1=True,
    mid_only_first=True,
    mid_only_last=True,
)

Run the model

In [5]:
model = DMACN(cfg)
model.fit(X, verbose_every=50)
labels = model.predict(save=True)
print("labels shape:", labels.shape)
print("labels: ", labels)

epoch    1/500 [mid-only]  J=2.0546e+03  J1=1.8900e+03  J2=5.5568e-06  J3=1.6455e+02  omega_sum=1.0000
epoch   50/500 [multilayer]  J=6.1990e+02  J1=4.6018e+02  J2=1.4761e-06  J3=1.5973e+02  omega_sum=1.0000
epoch  100/500 [multilayer]  J=5.7861e+02  J1=4.2599e+02  J2=1.1114e-05  J3=1.5262e+02  omega_sum=1.0000
epoch  150/500 [multilayer]  J=3.4727e+02  J1=1.9876e+02  J2=7.4832e-05  J3=1.4851e+02  omega_sum=1.0000
epoch  200/500 [multilayer]  J=3.2915e+02  J1=1.8730e+02  J2=9.4681e-05  J3=1.4185e+02  omega_sum=1.0000
epoch  250/500 [multilayer]  J=3.2041e+02  J1=1.8498e+02  J2=1.0447e-04  J3=1.3543e+02  omega_sum=1.0000
epoch  300/500 [multilayer]  J=3.1363e+02  J1=1.8432e+02  J2=1.1725e-04  J3=1.2931e+02  omega_sum=1.0000
epoch  350/500 [multilayer]  J=3.0681e+02  J1=1.8334e+02  J2=1.3167e-04  J3=1.2347e+02  omega_sum=1.0000
epoch  400/500 [multilayer]  J=3.0191e+02  J1=1.8400e+02  J2=1.4417e-04  J3=1.1791e+02  omega_sum=1.0000
epoch  450/500 [multilayer]  J=2.9446e+02  J1=1.8182e+02 

### UMAP

In [6]:
from UMAP import UMAP

C:\Users\oddar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\oddar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


ImportError: cannot import name 'UMAP' from 'UMAP' (c:\Users\oddar\OneDrive - NTNU\Documents\Universitet\Masteroppgåve\Master-Thesis-Autoencoder-and-UMAP-clustering-on-functional-connectivity\UMAP.py)

### HDBSCAN
Perform HDBSCAN on the data

In [ ]:
from HDBSCAN import hdbscan_clustering

hdbscan_clustering(Functional_connectivity_matrix=Functional_connectivity_matrix, save_labels=True)

### Evaluate the clusters
Evaluate the clusters using simple methods: Silhouette coefficient, Davies-Bouldin score and Calinski-Harabasz score

In [ ]:
from Evaluate_models import evaluate_clustering
import os

# Define where to find the labels 
labels_path = os.fsencode("Clusters")
evaluate_clustering(Functional_connectivity_matrix=Functional_connectivity_matrix, labels_path=labels_path)